# Análise exploratória – WIN e WDO (candles de 1 minuto)

**Disciplina:** Data Science no Mercado Financeiro

Este notebook faz **apenas análise descritiva** das bases `WIN$N_M1.csv` (mini índice) e `WDO$N_M1.csv` (mini dólar). Não há nenhum modelo de previsão aqui. O objetivo é entender os dados antes de modelar.

| Seção | O que observar |
|---|---|
| 1. Carregamento | Ler os arquivos do MetaTrader |
| 2. Qualidade dos dados | Nulos, duplicatas, candles inconsistentes, horários, coluna SPREAD |
| 3. Buracos na base | Períodos sem dados |
| 4. Visão geral do preço | Trajetória dos dois ativos |
| 5. Retornos | Estatísticas por timeframe, distribuição, caudas pesadas |
| 6. Eventos extremos | Maiores movimentos de 1 minuto |
| 7. Gaps de abertura e rolagem | Salto entre pregões e efeito da troca de contrato |
| 8. Volatilidade | Volatilidade ao longo do tempo e *volatility clustering* |
| 9. Padrões intradiários e semanais | Volatilidade e volume ao longo do dia e da semana |
| 10. Volume × movimento | Volume alto acompanha movimento grande? |
| 11. Autocorrelação e estacionariedade | ACF, teste ADF, razão de variâncias |
| 12. Direção dos candles | Uma alta tende a ser seguida de outra alta? |
| 13. Relação WIN × WDO | Correlação, estabilidade no tempo, lead-lag |
| 14. Resumo | Principais achados |

**Como usar no Colab:** coloque os dois CSVs numa pasta do Google Drive (por exemplo `MyDrive/data_science`) e ajuste `PASTA` na célula abaixo. A célula procura os arquivos em subpastas também.

## 1. Carregamento

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import acf, adfuller
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA = '/content/drive/MyDrive/data_science'   # <- ajuste para a pasta onde estão os CSVs
except ImportError:
    PASTA = '.'                                      # rodando fora do Colab

def achar(padrao):
    encontrados = glob.glob(f'{PASTA}/**/{padrao}', recursive=True)
    assert encontrados, f'Nenhum arquivo {padrao} encontrado em {PASTA}'
    return encontrados[0]

ARQUIVOS = {'WIN': achar('WIN*_M1.csv'), 'WDO': achar('WDO*_M1.csv')}
print(ARQUIVOS)

plt.rcParams.update({'figure.figsize': (14, 5), 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
CORES = {'WIN': '#1F5FA8', 'WDO': '#C0661A'}
pd.set_option('display.width', 200)

In [ ]:
def carregar(caminho):
    # Formato MetaTrader: separado por TAB, cabeçalhos entre < >
    df = pd.read_csv(caminho, sep='\t')
    df.columns = [c.strip('<>').lower() for c in df.columns]
    df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='%Y.%m.%d %H:%M:%S')
    return df.drop(columns=['date', 'time']).set_index('datetime').sort_index()

dados = {nome: carregar(arq) for nome, arq in ARQUIVOS.items()}
for nome, df in dados.items():
    print(f'{nome}: {len(df):,} candles | {df.index.min()} → {df.index.max()}')
    display(df.head(3))

## 2. Qualidade dos dados

Checagens básicas antes de qualquer análise:
- **Nulos e timestamps duplicados.**
- **Candles inconsistentes:** a máxima precisa ser ≥ abertura e fechamento; a mínima, ≤ as duas.
- **Candles por pregão e horários:** o pregão tem em torno de 550 minutos. Pregões curtos indicam dias atípicos (véspera de feriado, Quarta de Cinzas) ou falhas na base.

In [ ]:
def qualidade(df):
    dia = df.index.normalize()
    return pd.Series({
        'candles': f'{len(df):,}',
        'pregões': dia.nunique(),
        'início': df.index.min().date(),
        'fim': df.index.max().date(),
        'valores nulos': int(df.isna().sum().sum()),
        'timestamps duplicados': int(df.index.duplicated().sum()),
        'high < max(open, close)': int((df['high'] < df[['open', 'close']].max(axis=1)).sum()),
        'low > min(open, close)': int((df['low'] > df[['open', 'close']].min(axis=1)).sum()),
        'candles com volume 0': int((df['vol'] == 0).sum()),
        'mediana de candles por pregão': int(df.groupby(dia).size().median()),
    })

display(pd.DataFrame({nome: qualidade(df) for nome, df in dados.items()}))

In [ ]:
fig, axes = plt.subplots(1, 2)
for ax, (nome, df) in zip(axes, dados.items()):
    df.groupby(df.index.normalize()).size().hist(bins=60, ax=ax, color=CORES[nome])
    ax.set_title(f'{nome}: candles por pregão')
    ax.set_xlabel('candles no dia'); ax.set_ylabel('nº de pregões')
plt.tight_layout(); plt.show()

for nome, df in dados.items():
    horarios = df.index.to_series().groupby(df.index.normalize())
    abertura = horarios.min().dt.strftime('%H:%M').value_counts().head(3).to_dict()
    fechamento = horarios.max().dt.strftime('%H:%M').value_counts().head(3).to_dict()
    print(f'{nome} | primeiro candle mais comum: {abertura} | último candle mais comum: {fechamento}')

**O que observar:** o último candle do dia alterna entre dois horários (≈ 17:55 e ≈ 18:25). A B3 muda o horário de negociação ao longo do ano, acompanhando o horário de verão dos EUA. Isso importa para análises por horário do dia.

In [ ]:
# A coluna SPREAD é confiável?
for nome, df in dados.items():
    print(nome, 'SPREAD mediano por ano:', df.groupby(df.index.year)['spread'].median().to_dict())

**O que observar:** no WDO o SPREAD muda de escala em 2023 (de 500 para 1). Isso indica mudança na forma como a plataforma registra o campo, não no mercado. **A coluna SPREAD não deve ser usada** sem tratamento.

## 3. Buracos na base

O ideal é ter um pregão em todo dia útil. Intervalos de mais de 4 dias corridos entre dois pregões consecutivos não são fins de semana nem feriados comuns: indicam **dados faltando**.

In [ ]:
pregoes = {nome: pd.DatetimeIndex(df.index.normalize().unique()) for nome, df in dados.items()}

for nome, dias in pregoes.items():
    intervalo = pd.Series((dias[1:] - dias[:-1]).days, index=dias[1:])
    buracos = intervalo[intervalo > 5]
    print(f'\n{nome}: intervalos de mais de 5 dias corridos entre pregões')
    display(pd.DataFrame({'pregão anterior': [dias[dias.get_loc(d) - 1].date() for d in buracos.index],
                          'pregão seguinte': buracos.index.date,
                          'dias corridos': buracos.values}))

fig, ax = plt.subplots()
for nome, dias in pregoes.items():
    pd.Series(1, index=dias).resample('ME').sum().plot(ax=ax, color=CORES[nome], label=nome, marker='o', ms=3)
ax.set_title('Pregões por mês (o normal é ~20–22)'); ax.set_ylabel('pregões'); ax.legend()
plt.show()

**O que observar:** há um buraco grande em **2023** (jun → out no WIN, jun → set no WDO: de 3,5 a 4,5 meses sem dados) e buracos menores em **nov/2021** e **jul/2022**. Consequências:
- Qualquer retorno calculado "por cima" de um buraco mistura semanas ou meses de movimento num único ponto.
- Médias anuais de 2023 ficam baseadas em menos dias.
- Nas próximas seções os retornos são calculados **só entre candles/pregões consecutivos**.

## 4. Visão geral do preço

In [ ]:
fech_diario = {nome: df['close'].resample('D').last().dropna() for nome, df in dados.items()}

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
for ax, (nome, s) in zip(axes[:2], fech_diario.items()):
    ax.plot(s, color=CORES[nome], lw=1)
    ax.set_title(f'{nome}: fechamento diário (série contínua $N)'); ax.set_ylabel('pontos')

inicio = max(s.index.min() for s in fech_diario.values())
fim = min(s.index.max() for s in fech_diario.values())
for nome, s in fech_diario.items():
    trecho = s[inicio:fim]
    axes[2].plot(trecho / trecho.iloc[0] * 100, color=CORES[nome], lw=1, label=nome)
axes[2].set_title(f'Base 100 em {inicio.date()} (período comum aos dois ativos)'); axes[2].legend()
plt.tight_layout(); plt.show()

**O que observar:** os dois preços **não são estacionários**: têm tendências longas e não voltam para uma média fixa. Por isso as análises seguintes usam **retornos**, e não preços. Os trechos retos no gráfico são os buracos da Seção 3.

## 5. Retornos

Usamos o **log-retorno** $r_t = \ln(P_t / P_{t-1})$, calculado **apenas dentro do mesmo pregão**. Assim, o salto da noite (e os buracos da base) não contaminam os retornos intradiários.

Comparamos vários timeframes: 1, 5, 15 e 60 minutos, e o retorno do pregão (abertura → fechamento do dia).

In [ ]:
AGG = {'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last', 'tickvol': 'sum', 'vol': 'sum'}

def reamostrar(df, regra):
    return df.resample(regra).agg(AGG).dropna()

def ret_intradia(close):
    r = np.log(close).diff()
    dia = pd.Series(close.index.date, index=close.index)
    return r[dia.eq(dia.shift())].dropna()

def ret_pregao(df):
    g = df.groupby(df.index.normalize())
    return np.log(g['close'].last() / g['open'].first())

FREQS = ['1min', '5min', '15min', '60min']
barras = {nome: {f: (df if f == '1min' else reamostrar(df, f)) for f in FREQS} for nome, df in dados.items()}
ret = {nome: {f: ret_intradia(barras[nome][f]['close']) for f in FREQS} for nome in dados}
for nome, df in dados.items():
    ret[nome]['pregão'] = ret_pregao(df)

In [ ]:
def resumo(r):
    return pd.Series({
        'observações': len(r),
        'média (%)': r.mean() * 100,
        'desvio-padrão (%)': r.std() * 100,
        'assimetria': stats.skew(r),
        'curtose (excesso)': stats.kurtosis(r),
        'mínimo (%)': r.min() * 100,
        'máximo (%)': r.max() * 100,
        '% de alta': (r > 0).mean() * 100,
        '% de baixa': (r < 0).mean() * 100,
        '% sem variação': (r == 0).mean() * 100,
    })

tabela_ret = pd.concat({nome: pd.DataFrame({f: resumo(r) for f, r in rs.items()}) for nome, rs in ret.items()}, axis=1)
display(tabela_ret.round(3))

**Como ler:**
- **Média ≈ 0** em todos os timeframes: não há "deriva" previsível no curto prazo.
- **Curtose (excesso) muito acima de 0:** caudas pesadas. Movimentos extremos são muito mais frequentes do que numa distribuição Normal. A curtose diminui conforme o timeframe aumenta.
- **% sem variação:** relevante no WDO de 1 minuto. Se no futuro o alvo for a direção, essa classe precisa de tratamento.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, nome in enumerate(dados):
    r = ret[nome]['15min'] * 100
    lo, hi = r.quantile([0.001, 0.999])
    ax = axes[i, 0]
    ax.hist(r.clip(lo, hi), bins=150, density=True, color=CORES[nome], alpha=0.7, label='retornos observados')
    x = np.linspace(lo, hi, 400)
    ax.plot(x, stats.norm.pdf(x, r.mean(), r.std()), 'k--', label='Normal (mesma média e desvio)')
    ax.set_title(f'{nome}: distribuição dos retornos de 15 min (%)'); ax.legend()
    stats.probplot(r, dist='norm', plot=axes[i, 1])
    axes[i, 1].set_title(f'{nome}: QQ-plot contra a Normal (15 min)')
plt.tight_layout(); plt.show()

**O que observar:** o histograma é mais "pontudo" no centro e tem caudas mais longas que a Normal. No QQ-plot, os pontos se afastam da reta nas pontas: é a assinatura das **caudas pesadas**.

## 6. Eventos extremos

In [ ]:
for nome in dados:
    r = ret[nome]['1min']
    z = (r - r.mean()) / r.std()
    observado = (z.abs() > 4).mean()
    esperado = 2 * stats.norm.sf(4)
    print(f'{nome}: |retorno| > 4 desvios em {observado:.4%} dos minutos; '
          f'a Normal preveria {esperado:.4%} ({observado / esperado:,.0f}x menos)')

for nome in dados:
    r = ret[nome]['1min']
    top = r.abs().nlargest(10).index
    print(f'\n{nome}: 10 maiores movimentos de 1 minuto')
    display(pd.DataFrame({'retorno (%)': (r[top] * 100).round(2), 'dia da semana': top.day_name()}))

**O que observar:** as datas dos maiores movimentos costumam coincidir com eventos (decisões de juros, dados de inflação, notícias políticas, choques externos). Pesquise algumas datas: é uma boa discussão para o trabalho.

## 7. Gaps de abertura e rolagem de contratos

**Gap de abertura** = abertura do pregão ÷ fechamento do pregão anterior (em log). Só consideramos pregões **consecutivos** (no máximo 4 dias corridos de distância) para não contar os buracos da base.

**Rolagem:** `$N` é uma série contínua. No vencimento, a plataforma troca o contrato antigo pelo novo, e a diferença de preço entre eles aparece como um gap artificial.
- **WIN** vence nos meses pares, na quarta-feira mais próxima do dia 15.
- **WDO** vence todo mês, no primeiro dia útil.

In [ ]:
def gaps_abertura(df):
    g = df.groupby(df.index.normalize())
    abertura, fech_anterior = g['open'].first(), g['close'].last().shift()
    dias = abertura.index.to_series()
    consecutivo = (dias - dias.shift()).dt.days <= 4
    return np.log(abertura / fech_anterior)[consecutivo].dropna()

def vencimentos_win(anos):
    datas = []
    for ano in anos:
        for mes in range(2, 13, 2):
            dia15 = pd.Timestamp(ano, mes, 15)
            quartas = pd.date_range(dia15 - pd.Timedelta(days=3), dia15 + pd.Timedelta(days=3), freq='W-WED')
            datas.append(quartas[0])
    return pd.DatetimeIndex(datas)

gaps = {nome: gaps_abertura(df) for nome, df in dados.items()}
venc_win = vencimentos_win(range(2020, 2027))

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, (nome, g) in zip(axes, gaps.items()):
    ax.bar(g.index, g * 100, width=2, color=CORES[nome])
    ax.set_title(f'{nome}: gap de abertura (%)'); ax.set_ylabel('%')
for v in venc_win:
    axes[0].axvline(v, color='gray', lw=0.6, ls=':')
axes[0].set_title('WIN: gap de abertura (%) – linhas pontilhadas = vencimentos do WIN')
plt.tight_layout(); plt.show()

In [ ]:
g = gaps['WIN']
em_venc = g.index.isin(venc_win)
print('WIN – gap médio em dia de vencimento: %.2f%%  (n = %d)' % (g[em_venc].mean() * 100, em_venc.sum()))
print('WIN – gap médio nos outros dias:       %.2f%%  (n = %d)' % (g[~em_venc].mean() * 100, (~em_venc).sum()))

g = gaps['WDO']
primeiro_do_mes = g.index.to_series().groupby(g.index.to_period('M')).transform('min') == g.index.to_series()
print('\nWDO – |gap| médio no 1º pregão do mês: %.2f%%' % (g[primeiro_do_mes.values].abs().mean() * 100))
print('WDO – |gap| médio nos outros dias:     %.2f%%' % (g[~primeiro_do_mes.values].abs().mean() * 100))

for nome, g in gaps.items():
    print(f'\n{nome}: desvio-padrão do gap = {g.std() * 100:.2f}% | 5 maiores gaps:')
    top = g.abs().nlargest(5).index
    display(pd.DataFrame({'gap (%)': (g[top] * 100).round(2), 'dia da semana': top.day_name()}))

**O que observar:**
- No WIN, os dias de vencimento têm gap médio **positivo** e bem maior que nos outros dias. É a rolagem: o contrato seguinte é negociado com prêmio (juros de ~2 meses). Esse salto **não é retorno real**.
- No WDO o efeito é menor, porque o vencimento é mensal e o prêmio entre um mês e outro é pequeno.
- Para qualquer análise de retorno diário (fechamento a fechamento), esses dias precisam ser removidos ou ajustados.

## 8. Volatilidade

**Volatilidade realizada diária** = raiz da soma dos quadrados dos retornos de 1 minuto do pregão, anualizada por $\sqrt{252}$. É uma medida de "quanto o preço sacudiu" em cada dia, sem depender do gap noturno.

In [ ]:
vol_realizada = {nome: np.sqrt((ret[nome]['1min'] ** 2).groupby(ret[nome]['1min'].index.normalize()).sum()) * np.sqrt(252) * 100
                 for nome in dados}

fig, ax = plt.subplots(figsize=(14, 5))
for nome, v in vol_realizada.items():
    ax.plot(v, color=CORES[nome], alpha=0.25, lw=0.8)
    ax.plot(v.rolling(20).mean(), color=CORES[nome], lw=2, label=f'{nome} (média de 20 pregões)')
ax.set_title('Volatilidade realizada diária, anualizada (%)'); ax.set_ylabel('% ao ano'); ax.legend()
plt.show()

anual = pd.DataFrame({
    (nome, col): serie
    for nome, df in dados.items()
    for col, serie in {
        'pregões': df.groupby(df.index.year).apply(lambda x: x.index.normalize().nunique()),
        'vol. realizada média (% a.a.)': vol_realizada[nome].groupby(vol_realizada[nome].index.year).mean(),
        'volume médio por pregão (mil)': df['vol'].groupby(df.index.normalize()).sum().groupby(lambda d: d.year).mean() / 1e3,
    }.items()
})
display(anual.round(1))

In [ ]:
# Volatility clustering: o retorno quase não tem memória, mas o tamanho do movimento tem
fig, axes = plt.subplots(1, 2)
for ax, nome in zip(axes, dados):
    r = ret[nome]['15min']
    ax.plot(range(1, 101), acf(r, nlags=100, fft=True)[1:], color='gray', label='retorno r')
    ax.plot(range(1, 101), acf(r.abs(), nlags=100, fft=True)[1:], color=CORES[nome], label='|r| (tamanho do movimento)')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{nome}: autocorrelação de r e |r| (15 min)'); ax.set_xlabel('defasagem (candles de 15 min)'); ax.legend()
plt.tight_layout(); plt.show()

**O que observar:**
- A volatilidade **muda muito ao longo do tempo** e vem em "ondas": períodos calmos e períodos agitados (*volatility clustering*).
- A autocorrelação do retorno (cinza) fica perto de zero, mas a de **|r|** é positiva e persistente. A direção é difícil de prever; o **tamanho** do movimento, nem tanto.
- Os picos regulares na curva de |r| aparecem a cada ~37 candles de 15 minutos, ou seja, **um pregão**: é o padrão intradiário da próxima seção.

## 9. Padrões intradiários e semanais

In [ ]:
def perfil_intradiario(nome, minutos=15):
    r = ret[nome]['1min']
    df = dados[nome].loc[r.index]
    faixa = (r.index.hour * 60 + r.index.minute) // minutos * minutos
    rotulo = [f'{m // 60:02d}:{m % 60:02d}' for m in faixa]
    return pd.DataFrame({'|r| médio (bp)': r.abs().values * 1e4, 'volume médio': df['vol'].values,
                         'faixa': rotulo}).groupby('faixa').agg(['mean', 'size'])

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for i, nome in enumerate(dados):
    p = perfil_intradiario(nome)
    p = p[p[('volume médio', 'size')] > 0.2 * p[('volume médio', 'size')].max()]   # tira faixas raras
    axes[0, i].bar(p.index, p[('|r| médio (bp)', 'mean')], color=CORES[nome])
    axes[0, i].set_title(f'{nome}: tamanho médio do movimento por horário (bp por minuto)')
    axes[1, i].bar(p.index, p[('volume médio', 'mean')], color=CORES[nome], alpha=0.7)
    axes[1, i].set_title(f'{nome}: volume médio por minuto, por horário')
    for ax in axes[:, i]:
        ax.tick_params(axis='x', rotation=90, labelsize=8)
plt.tight_layout(); plt.show()

In [ ]:
DIAS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
NOMES_DIAS = ['seg', 'ter', 'qua', 'qui', 'sex']

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, nome in zip(axes, dados):
    r = ret[nome]['1min']
    tabela = (r.abs() * 1e4).groupby([r.index.day_name(), r.index.hour]).mean().unstack().reindex(DIAS)
    im = ax.imshow(tabela.values, aspect='auto', cmap='viridis')
    ax.set_yticks(range(5)); ax.set_yticklabels(NOMES_DIAS)
    ax.set_xticks(range(len(tabela.columns))); ax.set_xticklabels([f'{h}h' for h in tabela.columns])
    ax.set_title(f'{nome}: |r| médio de 1 min (bp) por dia × hora'); ax.grid(False)
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

semana = pd.concat({nome: (ret[nome]['pregão'] * 100).groupby(ret[nome]['pregão'].index.day_name())
                    .agg(['mean', 'std', 'count']).reindex(DIAS) for nome in dados}, axis=1)
semana.index = NOMES_DIAS
print('Retorno do pregão (abertura → fechamento, %) por dia da semana')
display(semana.round(3))

**O que observar:**
- **Perfil decrescente (formato de "L")**: a volatilidade e o volume são maiores na abertura e na primeira metade da manhã e vão caindo ao longo do dia. O fim do pregão quase não volta a acelerar. Volume e volatilidade andam juntos.
- Horários fixos com picos isolados costumam ser **divulgações de dados** (por exemplo, dados dos EUA às 9:30 ou 10:30 de Brasília) e a abertura do mercado americano.
- O retorno médio por dia da semana é pequeno perto do desvio-padrão: diferenças entre os dias provavelmente não são significativas.

## 10. Volume × movimento

In [ ]:
fig, axes = plt.subplots(1, 2)
for ax, nome in zip(axes, dados):
    b = barras[nome]['15min'].copy()
    b['abs_ret'] = np.log(b['close']).diff().abs() * 100
    b = b.loc[ret[nome]['15min'].index]
    b['decil de volume'] = pd.qcut(b['vol'], 10, labels=range(1, 11))
    medias = b.groupby('decil de volume', observed=True)['abs_ret'].mean()
    ax.bar(medias.index.astype(str), medias.values, color=CORES[nome])
    rho = stats.spearmanr(b['vol'], b['abs_ret']).statistic
    ax.set_title(f'{nome}: |retorno| médio por decil de volume (15 min)\ncorrelação de Spearman = {rho:.2f}')
    ax.set_xlabel('decil de volume (1 = menor, 10 = maior)'); ax.set_ylabel('|retorno| médio (%)')
plt.tight_layout(); plt.show()

**O que observar:** candles com mais volume têm movimentos bem maiores. Volume é um bom **indicador da intensidade** do movimento, mas não diz a **direção**.

## 11. Autocorrelação e estacionariedade

- **ACF dos retornos:** quanto o retorno de agora se parece com o de *k* candles atrás. As faixas cinza são o intervalo de 95% de um ruído branco.
- **Teste ADF:** hipótese nula = a série tem raiz unitária (não é estacionária). p-valor < 0,05 → estacionária.
- **Razão de variâncias:** VR(k) = variância do retorno de *k* minutos ÷ (*k* × variância do retorno de 1 minuto). Num passeio aleatório, VR = 1. VR < 1 indica reversão à média; VR > 1, tendência (momentum).

In [ ]:
fig, axes = plt.subplots(1, 2)
for ax, nome in zip(axes, dados):
    r = ret[nome]['1min']
    valores = acf(r, nlags=30, fft=True)[1:]
    ax.bar(range(1, 31), valores, color=CORES[nome])
    banda = 1.96 / np.sqrt(len(r))
    ax.axhspan(-banda, banda, color='gray', alpha=0.3)
    ax.set_title(f'{nome}: ACF dos retornos de 1 min'); ax.set_xlabel('defasagem (minutos)')
    print(f'{nome}: autocorrelação no lag 1 = {valores[0]:.4f}')
plt.tight_layout(); plt.show()

In [ ]:
linhas = []
for nome, df in dados.items():
    log_preco = np.log(df['close'].resample('D').last().dropna())
    for serie, x in [('log-preço diário', log_preco), ('retorno do pregão', ret[nome]['pregão']),
                     ('retorno 60 min', ret[nome]['60min'])]:
        estat, p = adfuller(x, autolag='AIC')[:2]
        linhas.append({'ativo': nome, 'série': serie, 'estatística ADF': round(estat, 2), 'p-valor': round(p, 4),
                       'conclusão': 'estacionária' if p < 0.05 else 'NÃO estacionária'})
display(pd.DataFrame(linhas))

In [ ]:
vr = {}
for nome in dados:
    var1 = ret[nome]['1min'].var()
    vr[nome] = {k: ret[nome][f'{k}min'].var() / (k * var1) for k in [5, 15, 60]}
display(pd.DataFrame(vr).rename_axis('k (minutos)').round(3))

**O que observar:**
- A autocorrelação dos retornos é minúscula: o passado diz muito pouco sobre o próximo retorno.
- O ADF confirma: **preço não é estacionário, retorno é**. É por isso que se modela retorno.
- A razão de variâncias mostra se, ao agregar minutos, os movimentos se somam (VR ≈ 1), se compensam (VR < 1) ou se reforçam (VR > 1).

## 12. Direção dos candles

In [ ]:
for nome in dados:
    r = ret[nome]['1min']
    direcao = np.sign(r).map({1.0: 'alta', -1.0: 'baixa', 0.0: 'parado'})
    tabela = pd.crosstab(direcao.shift().rename('candle anterior'), direcao.rename('candle seguinte'), normalize='index') * 100
    print(f'\n{nome}: probabilidade (%) da direção do candle seguinte, dada a direção do anterior (1 min)')
    display(tabela.round(1))

**O que observar:** se o mercado fosse totalmente aleatório, cada linha seria igual à proporção geral de altas e baixas. Diferenças pequenas (1–3 pontos percentuais) são comuns no minuto a minuto e podem vir do vaivém entre a melhor compra e a melhor venda (*bid-ask bounce*), não de uma oportunidade real.

## 13. Relação WIN × WDO

Dólar e bolsa costumam andar em sentidos opostos: em momentos de aversão a risco o dólar sobe e o índice cai. Aqui medimos:
1. A correlação **no mesmo candle**, em vários timeframes.
2. Se essa correlação é **estável no tempo**.
3. Se um ativo **antecipa** o outro (*lead-lag*).

In [ ]:
linhas = {}
for f in FREQS + ['pregão']:
    par = pd.concat({'WIN': ret['WIN'][f], 'WDO': ret['WDO'][f]}, axis=1, join='inner')
    linhas[f] = {'observações em comum': len(par), 'correlação': par['WIN'].corr(par['WDO'])}
display(pd.DataFrame(linhas).T.round(3))

par_dia = pd.concat({'WIN': ret['WIN']['pregão'], 'WDO': ret['WDO']['pregão']}, axis=1, join='inner')
fig, ax = plt.subplots()
ax.plot(par_dia['WIN'].rolling(60).corr(par_dia['WDO']), color='#5B3F8C')
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Correlação móvel de 60 pregões entre os retornos diários (abertura → fechamento) do WIN e do WDO')
plt.show()

In [ ]:
par = pd.concat({'WIN': ret['WIN']['1min'], 'WDO': ret['WDO']['1min']}, axis=1, join='inner')
defasagens = range(-10, 11)
cc = [par['WIN'].corr(par['WDO'].shift(k)) for k in defasagens]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].bar(defasagens, cc, color=['#5B3F8C' if k == 0 else 'gray' for k in defasagens])
axes[0].set_title('Correlação entre WIN(t) e WDO(t − k), candles de 1 min')
axes[0].set_xlabel('k > 0: WDO do passado × WIN de agora   |   k < 0: WIN do passado × WDO de agora')

p15 = pd.concat({'WIN': ret['WIN']['15min'], 'WDO': ret['WDO']['15min']}, axis=1, join='inner') * 100
axes[1].scatter(p15['WDO'], p15['WIN'], s=2, alpha=0.15, color='#5B3F8C')
axes[1].set_xlim(p15['WDO'].quantile([0.001, 0.999])); axes[1].set_ylim(p15['WIN'].quantile([0.001, 0.999]))
axes[1].set_xlabel('retorno do WDO (%)'); axes[1].set_ylabel('retorno do WIN (%)')
axes[1].set_title('Retornos de 15 min: WIN × WDO')
plt.tight_layout(); plt.show()
print('Correlação por defasagem k:', {k: round(float(v), 4) for k, v in zip(defasagens, cc) if abs(k) <= 3})

**O que observar:**
- A correlação negativa no mesmo candle é forte e fica mais forte em timeframes maiores.
- Ela **varia ao longo do tempo**: fica sempre negativa, mas oscila entre cerca de −0,2 e −0,8. Em quais períodos ela ficou mais forte? Vale relacionar com eventos (eleições, choques de juros).
- No lead-lag, praticamente toda a correlação está em **k = 0**. Um ativo quase não antecipa o outro: a informação chega aos dois ao mesmo tempo.

## 14. Resumo dos achados

Valores obtidos com as bases fornecidas pelo professor.

| Tema | Achado | Implicação |
|---|---|---|
| Qualidade | Sem nulos, duplicatas ou candles inconsistentes. ~565–570 candles por pregão | Base limpa no nível do candle |
| SPREAD | No WDO muda de escala em 2023 (500 → 1) | Não usar sem tratamento |
| Horário | Último candle alterna entre ≈ 17:55 e ≈ 18:25 | Cuidado ao comparar horários entre épocas do ano |
| **Buracos** | WIN sem dados de 02/06 a 17/10/2023; WDO de 02/06 a 20/09/2023; buracos menores em nov/2021 e jul/2022 | Não calcular retornos por cima dos buracos; 2023 tem menos pregões |
| Preço | Log-preço não estacionário (ADF p = 0,80 no WIN; 0,07 no WDO) | Trabalhar com retornos |
| Retornos | Média ≈ 0; desvio de 1 min: 0,046% (WIN) e 0,038% (WDO) | Sem deriva previsível no curto prazo |
| Caudas pesadas | Curtose de 22 (WIN) e 24 (WDO) em 1 min; movimentos > 4 desvios ~80x mais frequentes que na Normal | Hipótese de normalidade não vale; atenção a outliers |
| Minutos parados | 15,3% dos minutos do WDO e 5,5% do WIN sem variação | Classe "parado" relevante no WDO |
| **Rolagem** | Gap médio de **+1,64%** nos vencimentos do WIN, contra 0,00% nos outros dias | Retirar ou ajustar dias de rolagem em retornos diários |
| Volatilidade | Muda ao longo do tempo (WIN: ~20% a.a. em 2022, ~12% em 2024) e vem em ondas | *Volatility clustering*: |r| tem memória, r não |
| Intradiário | Volatilidade e volume altos na abertura e na manhã, caindo ao longo do dia | Padrão diário que se repete (~37 candles de 15 min) |
| Volume | Correlação de Spearman entre volume e tamanho do movimento: 0,47 (WIN) e 0,51 (WDO), em 15 min | Volume indica intensidade, não direção |
| Autocorrelação | Lag 1 de 1 min: −0,009 (WIN) e −0,034 (WDO) | Passado quase não explica o próximo retorno |
| Razão de variâncias | VR(60) = 0,89 (WIN) e 0,76 (WDO) | Leve reversão à média intradiária, mais forte no WDO |
| Direção | No WDO, depois de uma alta a chance de baixa (45,5%) supera a de alta (39,7%) | Coerente com o *bid-ask bounce* |
| **WIN × WDO** | Correlação de −0,45 (1 min) a −0,54 (diário); sempre negativa, oscilando de −0,2 a −0,8; lead-lag ≈ 0 | Relação forte no mesmo instante, mas nenhum antecipa o outro |

### Outras análises descritivas que podem ser feitas

- **Calendário de eventos:** marcar as datas de Copom, FOMC, payroll e IPCA e comparar a volatilidade nesses dias com a dos outros.
- **Volatilidade por regime:** separar períodos de alta e baixa volatilidade e comparar a correlação WIN × WDO em cada um.
- **Amplitude do candle (High − Low):** distribuição e relação com volume e horário.
- **Volume financeiro:** volume × preço, para comparar épocas com níveis de preço diferentes.
- **Fins de mês e vésperas de feriado:** comportamento em dias atípicos.
- **Teste de normalidade formal:** Jarque-Bera nos retornos de cada timeframe.
- **Comparação entre anos:** repetir as tabelas por ano para ver se os padrões são estáveis.